In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [9]:
file_path = 'C:/Users/mannu/2D Protien Folding/RS126.data.txt'
with open(file_path, 'r') as file:
       lines = file.readlines()

sequences, structures = [], []
for i in range(0, len(lines) - 1, 2):
    seq, struct = lines[i].strip(), lines[i+1].strip()
    if len(seq) == len(struct) and len(seq) > 0:
        sequences.append(seq)
        structures.append(struct)

In [10]:
window_size = 13
pad_length = window_size // 2
X_data, Y_data = [], []
for seq, struct in zip(sequences[:50], structures[:50]):
    padded_seq = ("X" * pad_length) + seq + ("X" * pad_length)
    for j in range(len(seq)):
        X_data.append(padded_seq[j : j + window_size])
        Y_data.append(struct[j])


In [15]:
alphabet = "ACDEFGHIKLMNPQRSTVWYX"
char_to_index = {char: idx for idx, char in enumerate(alphabet)}

# Shape: [Total_Rows, 13] of type torch.long
X_ints = torch.zeros(len(X_data), window_size, dtype=torch.long)
for row_idx, window in enumerate(X_data):
    for char_idx, char in enumerate(window):
        if char in char_to_index:
            X_ints[row_idx, char_idx] = char_to_index[char]

# 4. CONVERT TARGET SHAPES TO INTEGERS (C->0, E->1, H->2)
shape_mapping = {'C': 0, 'E': 1, 'H': 2}
Y_ints = [shape_mapping[shape] for shape in Y_data]
Y_tensor = torch.tensor(Y_ints, dtype=torch.long)

# 5. CREATE DATALOADER FOR MINI-BATCHES
dataset = TensorDataset(X_ints, Y_tensor)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

print("Data successfully prepared for Transformer!")
print(f"X_ints shape: {X_ints.shape}  <-- [Total_Rows, 13] (Integer Tokens)")
print(f"Y_tensor shape: {Y_tensor.shape}")


Data successfully prepared for Transformer!
X_ints shape: torch.Size([8289, 13])  <-- [Total_Rows, 13] (Integer Tokens)
Y_tensor shape: torch.Size([8289])


In [23]:
class MiniFoldTransformer(nn.Module):
    def __init__(self, vocab_size=21, window_size=13, d_model=128, nhead=4, num_layers=2, output_size=3, dropout=0.3):
        super(MiniFoldTransformer, self).__init__()
        
        self.window_size = window_size
        self.d_model = d_model
        
        # 1. Biological Embedding Layer: [Batch, 13] -> [Batch, 13, 64]
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
        
        # 2. Learned Positional Embeddings: Injects position order (0 through 12)
        self.pos_embedding = nn.Embedding(num_embeddings=window_size, embedding_dim=d_model)
        
        # 3. Transformer Encoder Block (Self-Attention + Feed Forward)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,  # 128 internal hidden neurons
            dropout=dropout,
            activation='relu',
            batch_first=True  # MANDATORY: Ensures shape is [Batch, Length, Features]
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 4. Final Classification Head (Evaluates the middle token -> C, E, or H)
        self.fc = nn.Linear(d_model, output_size)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # x shape entering: [Batch, 13] (integers)
        batch_size, seq_len = x.shape
        
        # 1. Calculate biological embeddings: [Batch, 13, 64]
        token_embeddings = self.embedding(x)
        
        # 2. Add positional information: [Batch, 13, 64]
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0).repeat(batch_size, 1)
        pos_embeddings = self.pos_embedding(positions)
        x = token_embeddings + pos_embeddings
        x = self.dropout(x)
        
        # 3. Pass through Self-Attention layers: [Batch, 13, 64]
        x = self.transformer_encoder(x)
        
        # 4. Extract ONLY the middle amino acid (Index 6): [Batch, 64]
        middle_idx = self.window_size // 2  # 13 // 2 = 6
        middle_token = x[:, middle_idx, :]
        
        # 5. Classify the middle token: [Batch, 3] (Logits for C, E, H)
        out = self.fc(middle_token)
        return out

print("MiniFoldTransformer class successfully defined!")

MiniFoldTransformer class successfully defined!


In [24]:
# HYPERPARAMETERS
VOCAB_SIZE = len(alphabet) # 21
WINDOW_SIZE = 13
D_MODEL = 64
NHEAD = 4
NUM_LAYERS = 2
OUTPUT_SIZE = 3

# Instantiate Model
model = MiniFoldTransformer(
    vocab_size=VOCAB_SIZE,
    window_size=WINDOW_SIZE,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    output_size=OUTPUT_SIZE,
    dropout=0.3
)

criterion = nn.CrossEntropyLoss()
# Transformers train best with a slightly lower learning rate (0.0005)
optimizer = optim.Adam(model.parameters(), lr=0.0005)

print("Transformer Model successfully initialized!")
print(model)

Transformer Model successfully initialized!
MiniFoldTransformer(
  (embedding): Embedding(21, 64)
  (pos_embedding): Embedding(13, 64)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=64, out_features=3, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [29]:
NUM_EPOCHS = 100

print("--- STARTING TRANSFORMER TRAINING ---")
for epoch in range(NUM_EPOCHS):
    model.train() # Enable Dropout
    running_loss = 0.0
    
    for batch_X, batch_Y in train_loader:
        # batch_X is shape [64, 13] (integers)
        
        # 1. Forward Pass (Self-Attention happens here!)
        outputs = model(batch_X) # Output shape: [64, 3]
        
        # 2. Calculate Loss
        loss = criterion(outputs, batch_Y)
        
        # 3. Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_loss = running_loss / len(train_loader)
    if(epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Average Loss: {avg_loss:.4f}")

print("--- TRAINING COMPLETE ---")

--- STARTING TRANSFORMER TRAINING ---
Epoch [1/100] | Average Loss: 0.7608
Epoch [10/100] | Average Loss: 0.7577
Epoch [20/100] | Average Loss: 0.7680
Epoch [30/100] | Average Loss: 0.7482
Epoch [40/100] | Average Loss: 0.7464
Epoch [50/100] | Average Loss: 0.7360
Epoch [60/100] | Average Loss: 0.7374
Epoch [70/100] | Average Loss: 0.7293
Epoch [80/100] | Average Loss: 0.7228
Epoch [90/100] | Average Loss: 0.7188
Epoch [100/100] | Average Loss: 0.7085
--- TRAINING COMPLETE ---


In [30]:
def predict_protein_structure_transformer(protein_seq, trained_model, window_size=13, alphabet="ACDEFGHIKLMNPQRSTVWYX"):
    trained_model.eval() # Turn off Dropout
    
    pad_length = window_size // 2
    padded_seq = ("X" * pad_length) + protein_seq + ("X" * pad_length)
    
    char_to_idx = {char: i for i, char in enumerate(alphabet)}
    protein_len = len(protein_seq)
    
    # 1. Build 2D integer tensor: [Protein_Length, 13]
    X_test_ints = torch.zeros(protein_len, window_size, dtype=torch.long)
    for i in range(protein_len):
        window = padded_seq[i : i + window_size]
        for char_idx, char in enumerate(window):
            if char in char_to_idx:
                X_test_ints[i, char_idx] = char_to_idx[char]
                
    # 2. Run Forward Pass with Self-Attention
    with torch.no_grad():
        outputs = trained_model(X_test_ints) # Shape: [Length, 3]
        predicted_indices = torch.argmax(outputs, dim=1)
        
    # 3. Translate integer indices back to letters
    int_to_shape = {0: 'C', 1: 'E', 2: 'H'}
    predicted_chars = [int_to_shape[int(idx.item())] for idx in predicted_indices]
    
    return "".join(predicted_chars)

# --- RUN INFERENCE TEST ---
test_input      = "SIPPEVKFNKPFVFLMIEQNTKSPLFMGKVVNPTQK"
expected_output = "CCCCEEECCCCEEEEEEECCCCCEEEEEEECCCCCC"

trans_pred = predict_protein_structure_transformer(test_input, model)

matches = sum(1 for p, e in zip(trans_pred, expected_output) if p == e)
accuracy = (matches / len(expected_output)) * 100

print("--- TRANSFORMER INFERENCE TEST RESULTS ---")
print(f"Input Sequence:   {test_input}")
print(f"Expected Ground:  {expected_output}")
print(f"Transformer Pred: {trans_pred}")
print(f"Accuracy:         {accuracy:.1f}% ({matches}/{len(expected_output)} correct amino acids)")

--- TRANSFORMER INFERENCE TEST RESULTS ---
Input Sequence:   SIPPEVKFNKPFVFLMIEQNTKSPLFMGKVVNPTQK
Expected Ground:  CCCCEEECCCCEEEEEEECCCCCEEEEEEECCCCCC
Transformer Pred: CCCCCEECCCCEEEEEEHCCCCCCEEEEEECCCCCC
Accuracy:         91.7% (33/36 correct amino acids)
